In [ ]:
!pip -q install kaggle

import os, shutil
from google.colab import files
files.upload()


os.makedirs("/root/.kaggle", exist_ok=True)
shutil.copy("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("Kaggle API ready")


Saving kaggle.json to kaggle (1).json
Kaggle API ready


In [ ]:
!kaggle -v


Kaggle API 1.7.4.5


In [ ]:
!kaggle datasets download -d elemento/nyc-yellow-taxi-trip-data -p /content --unzip


Dataset URL: https://www.kaggle.com/datasets/elemento/nyc-yellow-taxi-trip-data
License(s): U.S. Government Works
 99% 1.76G/1.78G [00:17<00:00, 220MB/s]
100% 1.78G/1.78G [00:17<00:00, 110MB/s]


In [ ]:
!ls -lah /content | head -n 50


total 6.9G
drwxr-xr-x 1 root root 4.0K Jan 31 07:41 .
drwxr-xr-x 1 root root 4.0K Jan 31 06:31 ..
drwxr-xr-x 4 root root 4.0K Jan 16 14:24 .config
-rw-r--r-- 1 root root   67 Jan 31 07:39 kaggle (1).json
-rw-r--r-- 1 root root   67 Jan 31 06:32 kaggle.json
drwxr-xr-x 1 root root 4.0K Jan 16 14:24 sample_data
-rw-r--r-- 1 root root 3.0M Jan 31 06:34 train_sample.csv
-rw-r--r-- 1 root root 1.1M Jan 31 06:34 xgb_taxi_model.pkl
-rw-r--r-- 1 root root 1.9G Jan 31 07:40 yellow_tripdata_2015-01.csv
-rw-r--r-- 1 root root 1.6G Jan 31 07:41 yellow_tripdata_2016-01.csv
-rw-r--r-- 1 root root 1.7G Jan 31 07:41 yellow_tripdata_2016-02.csv
-rw-r--r-- 1 root root 1.8G Jan 31 07:41 yellow_tripdata_2016-03.csv


In [ ]:
import glob
import pandas as pd

csv_files = sorted(glob.glob("/content/*.csv"))
print("Found CSV:", len(csv_files))
print("Example:", csv_files[:5])

path = csv_files[0]   # lấy file đầu tiên
df = pd.read_csv(path, nrows=20000)

df.to_csv("/content/train_sample.csv", index=False)
print("Sample saved:", "/content/train_sample.csv", df.shape)


Found CSV: 5
Example: ['/content/train_sample.csv', '/content/yellow_tripdata_2015-01.csv', '/content/yellow_tripdata_2016-01.csv', '/content/yellow_tripdata_2016-02.csv', '/content/yellow_tripdata_2016-03.csv']
Sample saved: /content/train_sample.csv (20000, 19)


In [ ]:
print(df.columns)
df.head(3)


Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'pickup_longitude',
       'pickup_latitude', 'RateCodeID', 'store_and_fwd_flag',
       'dropoff_longitude', 'dropoff_latitude', 'payment_type', 'fare_amount',
       'extra', 'mta_tax', 'tip_amount', 'tolls_amount',
       'improvement_surcharge', 'total_amount'],
      dtype='object')


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RateCodeID,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount
0,2,2015-01-15 19:05:39,2015-01-15 19:23:42,1,1.59,-73.993896,40.750111,1,N,-73.974785,40.750618,1,12.0,1.0,0.5,3.25,0.0,0.3,17.05
1,1,2015-01-10 20:33:38,2015-01-10 20:53:28,1,3.30,-74.001648,40.724243,1,N,-73.994415,40.759109,1,14.5,0.5,0.5,2.00,0.0,0.3,17.80
2,1,2015-01-10 20:33:38,2015-01-10 20:43:41,1,1.80,-73.963341,40.802788,1,N,-73.951820,40.824413,2,9.5,0.5,0.5,0.00,0.0,0.3,10.80


FEATURE ENGINEERING

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/train_sample.csv")

# parse datetime
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"])

# temporal features
df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
df["pickup_weekday"] = df["tpep_pickup_datetime"].dt.weekday
df["pickup_month"] = df["tpep_pickup_datetime"].dt.month
df["is_weekend"] = df["pickup_weekday"].isin([5, 6]).astype(int)

# rush hour flag (NYC typical)
df["is_rush_hour"] = df["pickup_hour"].isin([7,8,9,16,17,18]).astype(int)

df[[
    "pickup_hour",
    "pickup_weekday",
    "pickup_month",
    "is_weekend",
    "is_rush_hour"
]].head()


,pickup_hour,pickup_weekday,pickup_month,is_weekend,is_rush_hour
0,19,3,1,0,0
1,20,5,1,1,0
2,20,5,1,1,0
3,20,5,1,1,0
4,20,5,1,1,0


Spatial Features (Distance + Direction)

In [ ]:
import numpy as np

# Haversine distance (km)
def haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    r = 6371  # Earth radius (km)
    return c * r

# Manhattan distance proxy (km)
def manhattan_distance(lon1, lat1, lon2, lat2):
    return (
        haversine(lon1, lat1, lon2, lat1) +
        haversine(lon2, lat1, lon2, lat2)
    )

# Bearing (direction)
def bearing(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return np.degrees(np.arctan2(x, y))

# Apply spatial features
df["haversine_distance"] = haversine(
    df["pickup_longitude"], df["pickup_latitude"],
    df["dropoff_longitude"], df["dropoff_latitude"]
)

df["manhattan_distance"] = manhattan_distance(
    df["pickup_longitude"], df["pickup_latitude"],
    df["dropoff_longitude"], df["dropoff_latitude"]
)

df["bearing"] = bearing(
    df["pickup_longitude"], df["pickup_latitude"],
    df["dropoff_longitude"], df["dropoff_latitude"]
)

df[["haversine_distance", "manhattan_distance", "bearing"]].head()


,haversine_distance,manhattan_distance,bearing
0,1.610893,1.666327,87.986790
1,3.924552,4.486456,8.929682
2,2.592739,3.374324,21.954795
3,0.794628,1.087142,30.325198
4,3.544626,4.978638,-128.331214


FASTEST ROUTE

In [ ]:
import numpy as np

# --- Airport coordinates (approx) ---
JFK = (-73.7781, 40.6413)
LGA = (-73.8740, 40.7769)
EWR = (-74.1745, 40.6895)

def near_airport(lon, lat, airport_lon, airport_lat, thresh_km=2.0):
    d = haversine(lon, lat, airport_lon, airport_lat)
    return (d <= thresh_km).astype(int)

# flags: pickup/dropoff near airports
df["pickup_near_JFK"]  = near_airport(df["pickup_longitude"],  df["pickup_latitude"],  JFK[0], JFK[1], 2.0)
df["dropoff_near_JFK"] = near_airport(df["dropoff_longitude"], df["dropoff_latitude"], JFK[0], JFK[1], 2.0)

df["pickup_near_LGA"]  = near_airport(df["pickup_longitude"],  df["pickup_latitude"],  LGA[0], LGA[1], 2.0)
df["dropoff_near_LGA"] = near_airport(df["dropoff_longitude"], df["dropoff_latitude"], LGA[0], LGA[1], 2.0)

df["pickup_near_EWR"]  = near_airport(df["pickup_longitude"],  df["pickup_latitude"],  EWR[0], EWR[1], 2.0)
df["dropoff_near_EWR"] = near_airport(df["dropoff_longitude"], df["dropoff_latitude"], EWR[0], EWR[1], 2.0)

# airport any
df["pickup_airport"]  = (df["pickup_near_JFK"]  | df["pickup_near_LGA"]  | df["pickup_near_EWR"]).astype(int)
df["dropoff_airport"] = (df["dropoff_near_JFK"] | df["dropoff_near_LGA"] | df["dropoff_near_EWR"]).astype(int)

# --- Fastest-route proxy ---
# baseline speed (km/h) with penalties for rush hour & airport trips
base_speed = 22.0
rush_penalty = 0.75  # slower in rush hour
airport_penalty = 0.85  # slower near airports

speed = base_speed * (1 - df["is_rush_hour"]*(1-rush_penalty)) * (1 - (df["pickup_airport"]|df["dropoff_airport"])*(1-airport_penalty))

# avoid division by zero
speed = np.clip(speed, 5.0, None)

# estimated travel time in minutes (proxy)
df["eta_minutes_proxy"] = (df["manhattan_distance"] / speed) * 60.0

# interaction features often help tree models
df["dist_x_rush"] = df["manhattan_distance"] * df["is_rush_hour"]
df["dist_x_weekend"] = df["manhattan_distance"] * df["is_weekend"]

df[[
    "pickup_airport","dropoff_airport",
    "eta_minutes_proxy","dist_x_rush","dist_x_weekend"
]].head()


,pickup_airport,dropoff_airport,eta_minutes_proxy,dist_x_rush,dist_x_weekend
0,0,0,4.544527,0.0,0.000000
1,0,0,12.235789,0.0,4.486456
2,0,0,9.202703,0.0,3.374324
3,0,0,2.964931,0.0,1.087142
4,0,0,13.578104,0.0,4.978638


WEATHER

In [ ]:
df["pickup_date"] = df["tpep_pickup_datetime"].dt.date
df[["tpep_pickup_datetime", "pickup_date"]].head()


,tpep_pickup_datetime,pickup_date
0,2015-01-15 19:05:39,2015-01-15
1,2015-01-10 20:33:38,2015-01-10
2,2015-01-10 20:33:38,2015-01-10
3,2015-01-10 20:33:39,2015-01-10
4,2015-01-10 20:33:39,2015-01-10


In [ ]:
import pandas as pd
import numpy as np

dates = pd.date_range(
    start=df["pickup_date"].min(),
    end=df["pickup_date"].max(),
    freq="D"
)

weather = pd.DataFrame({
    "date": dates.date,
    "avg_temp": np.random.normal(5, 7, len(dates)),        # °C (winter NYC)
    "precip_mm": np.random.exponential(2, len(dates)),     # precipitation
})

weather["is_rain"] = (weather["precip_mm"] > 1.0).astype(int)
weather["is_snow"] = ((weather["avg_temp"] < 0) & (weather["precip_mm"] > 0.5)).astype(int)

weather.head()


,date,avg_temp,precip_mm,is_rain,is_snow
0,2015-01-01,8.439577,2.187558,1,0
1,2015-01-02,-0.796238,0.670208,0,1
2,2015-01-03,6.625323,2.016131,1,0
3,2015-01-04,2.816619,0.513247,0,0
4,2015-01-05,3.836378,5.015662,1,0


In [ ]:
df = df.merge(
    weather,
    left_on="pickup_date",
    right_on="date",
    how="left"
)

df[["avg_temp", "precip_mm", "is_rain", "is_snow"]].head()


,avg_temp,precip_mm,is_rain,is_snow
0,21.988976,1.294813,1,0
1,9.201697,0.200461,0,0
2,9.201697,0.200461,0,0
3,9.201697,0.200461,0,0
4,9.201697,0.200461,0,0


In [ ]:
feature_cols = [
    # temporal
    "pickup_hour","pickup_weekday","pickup_month",
    "is_weekend","is_rush_hour",

    # spatial
    "haversine_distance","manhattan_distance","bearing",

    # route proxy
    "pickup_airport","dropoff_airport","eta_minutes_proxy",
    "dist_x_rush","dist_x_weekend",

    # weather
    "avg_temp","precip_mm","is_rain","is_snow"
]

df[feature_cols].head()


,pickup_hour,pickup_weekday,pickup_month,is_weekend,is_rush_hour,haversine_distance,manhattan_distance,bearing,pickup_airport,dropoff_airport,eta_minutes_proxy,dist_x_rush,dist_x_weekend,avg_temp,precip_mm,is_rain,is_snow
0,19,3,1,0,0,1.610893,1.666327,87.986790,0,0,4.544527,0.0,0.000000,21.988976,1.294813,1,0
1,20,5,1,1,0,3.924552,4.486456,8.929682,0,0,12.235789,0.0,4.486456,9.201697,0.200461,0,0
2,20,5,1,1,0,2.592739,3.374324,21.954795,0,0,9.202703,0.0,3.374324,9.201697,0.200461,0,0
3,20,5,1,1,0,0.794628,1.087142,30.325198,0,0,2.964931,0.0,1.087142,9.201697,0.200461,0,0
4,20,5,1,1,0,3.544626,4.978638,-128.331214,0,0,13.578104,0.0,4.978638,9.201697,0.200461,0,0


In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split

# Target (ETA proxy)
y = df["eta_minutes_proxy"]

# Features
X = df[feature_cols]

# Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=False
)

# Model
model = XGBRegressor(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

model.fit(X_train, y_train)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=200,
             n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
import joblib
joblib.dump(model, "xgb_taxi_model.pkl")



['xgb_taxi_model.pkl']

In [ ]:
import joblib

model = joblib.load("xgb_taxi_model.pkl")
print(type(model))


<class 'xgboost.sklearn.XGBRegressor'>


In [ ]:
import pandas as pd
import numpy as np

def predict_duration(feature_dict):
    """feature_dict: dictionary of engineered features"""
    X_input = pd.DataFrame([feature_dict])[feature_cols]
    pred = model.predict(X_input)[0]
    return pred


In [ ]:
sample_features = df[feature_cols].iloc[0].to_dict()
pred_time = predict_duration(sample_features)

print("Predicted ETA (minutes):", pred_time)


Predicted ETA (minutes): 4.503868


In [ ]:
import os
os.environ["ORS_API_KEY"] = "eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjEzMDM2NWU4MmEzOTQ4NDhiNWJmNzAzMGNiNTM2MDY3IiwiaCI6Im11cm11cjY0In0="



In [ ]:
!pip install openrouteservice


In [ ]:
import openrouteservice
client = openrouteservice.Client(key=os.environ["ORS_API_KEY"])


In [ ]:
import openrouteservice
import os

client = openrouteservice.Client(
    key=os.environ["ORS_API_KEY"]
)

print("ORS client ready")


ORS client ready


In [ ]:
!pip install folium


In [ ]:
assert "df" in globals(), "Dataframe not loaded"


In [ ]:
import folium
import pandas as pd

sample = df.iloc[0]
pickup = (sample["pickup_longitude"], sample["pickup_latitude"])
dropoff = (sample["dropoff_longitude"], sample["dropoff_latitude"])

route = client.directions(
    coordinates=[pickup, dropoff],
    profile="driving-car",
    format="geojson"
)

geometry = route["features"][0]["geometry"]["coordinates"]
route_eta = route["features"][0]["properties"]["summary"]["duration"] / 60
distance_km = route["features"][0]["properties"]["summary"]["distance"] / 1000

features = {
    "pickup_hour": sample["pickup_hour"],
    "pickup_weekday": sample["pickup_weekday"],
    "pickup_month": sample["pickup_month"],
    "is_weekend": sample["is_weekend"],
    "is_rush_hour": sample["is_rush_hour"],
    "haversine_distance": distance_km,
    "manhattan_distance": distance_km * 1.2,
    "bearing": sample["bearing"],
    "pickup_airport": sample["pickup_airport"],
    "dropoff_airport": sample["dropoff_airport"],
    "eta_minutes_proxy": route_eta,
    "dist_x_rush": distance_km * sample["is_rush_hour"],
    "dist_x_weekend": distance_km * sample["is_weekend"],
    "avg_temp": sample["avg_temp"],
    "precip_mm": sample["precip_mm"],
    "is_rain": sample["is_rain"],
    "is_snow": sample["is_snow"]
}

import pandas as pd

X_input = pd.DataFrame([features])[feature_cols]
pred_eta = model.predict(X_input)[0]

print(f"Predicted ETA (minutes): {pred_eta:.2f}")
import folium

center_lat = (pickup[1] + dropoff[1]) / 2
center_lon = (pickup[0] + dropoff[0]) / 2

m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

folium.Marker(
    [pickup[1], pickup[0]],
    popup="Pickup Location",
    icon=folium.Icon(color="green")
).add_to(m)

folium.Marker(
    [dropoff[1], dropoff[0]],
    popup="Dropoff Location",
    icon=folium.Icon(color="red")
).add_to(m)

folium.PolyLine(
    [(lat, lon) for lon, lat in geometry],
    color="blue"
).add_to(m)

Predicted ETA (minutes): 5.54


In [ ]:
m